# System Metrics Notebook

This notebook monitors and analyzes system performance:
- Inference latency
- Frame synchronization metrics
- Tracking persistence
- Resource utilization

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import time
import cv2
import psutil

## 1. System Information

In [ ]:
import platform

print("System Information:")
print(f"  Platform: {platform.system()} {platform.release()}")
print(f"  Python: {platform.python_version()}")
print(f"  CPU: {platform.processor()}")
print(f"  CPU Cores: {psutil.cpu_count()}")
print(f"  RAM: {psutil.virtual_memory().total / 1e9:.1f} GB")

# Check for GPU
try:
    import torch
    print(f"  PyTorch: {torch.__version__}")
    print(f"  CUDA Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
except ImportError:
    print("  PyTorch not installed")

## 2. Inference Latency Analysis

In [ ]:
from ultralytics import YOLO

model_path = Path('../models/poultry-yolov12n-v1.pt')
if model_path.exists():
    model = YOLO(str(model_path))
    
    # Benchmark different image sizes
    sizes = [320, 480, 640, 800]
    results = []
    
    for size in sizes:
        img = np.random.randint(0, 255, (size, size, 3), dtype=np.uint8)
        
        # Warmup
        for _ in range(3):
            model(img, verbose=False)
        
        # Benchmark
        times = []
        for _ in range(20):
            start = time.time()
            model(img, verbose=False)
            times.append((time.time() - start) * 1000)
        
        results.append({
            'size': size,
            'avg_ms': np.mean(times),
            'std_ms': np.std(times),
            'fps': 1000 / np.mean(times)
        })
        print(f"Size {size}x{size}: {np.mean(times):.1f}ms ({1000/np.mean(times):.1f} FPS)")
    
    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax1.bar([str(r['size']) for r in results], [r['avg_ms'] for r in results])
    ax1.set_xlabel('Image Size')
    ax1.set_ylabel('Latency (ms)')
    ax1.set_title('Inference Latency vs Image Size')
    
    ax2.bar([str(r['size']) for r in results], [r['fps'] for r in results])
    ax2.set_xlabel('Image Size')
    ax2.set_ylabel('FPS')
    ax2.set_title('Throughput vs Image Size')
    ax2.axhline(y=30, color='r', linestyle='--', label='30 FPS target')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("Model not found")

## 3. Synchronization Simulation

In [ ]:
from src.core.synchronization import FrameSynchronizer, SyncConfig
from src.core.ingestion import TimestampedFrame

# Simulate frame synchronization with varying latency
config = SyncConfig(tolerance_ms=50.0, primary_camera='top')
sync = FrameSynchronizer(config)

# Simulate 100 frame pairs with varying latency
np.random.seed(42)
latencies = []
sync_successes = []

for i in range(100):
    base_time = i * 33.3  # 30 FPS
    
    # Primary camera - minimal latency
    top_time = base_time + np.random.normal(0, 5)
    
    # Secondary camera - higher latency (RTSP simulation)
    side_latency = np.random.normal(30, 15)  # Mean 30ms, std 15ms
    side_time = base_time + side_latency
    
    latencies.append(abs(top_time - side_time))
    
    # Add frames
    sync.add_frame(TimestampedFrame(
        frame=np.zeros((480, 640, 3), dtype=np.uint8),
        timestamp_ms=top_time, camera_name='top', frame_id=i
    ))
    sync.add_frame(TimestampedFrame(
        frame=np.zeros((480, 640, 3), dtype=np.uint8),
        timestamp_ms=side_time, camera_name='side', frame_id=i
    ))
    
    pair = sync.get_synced_pair()
    sync_successes.append(pair is not None)

stats = sync.get_sync_stats()

print("Synchronization Simulation Results:")
print(f"  Sync rate: {stats['sync_rate']*100:.1f}%")
print(f"  Frames synced: {stats['frames_synced']}")
print(f"  Frames dropped: {stats['frames_dropped']}")
print(f"  Avg latency diff: {np.mean(latencies):.1f}ms")

# Plot latency distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(latencies, bins=20, edgecolor='black')
ax1.axvline(x=50, color='r', linestyle='--', label='50ms tolerance')
ax1.set_xlabel('Frame Latency Difference (ms)')
ax1.set_ylabel('Count')
ax1.set_title('Latency Distribution')
ax1.legend()

cumsum = np.cumsum(sync_successes)
ax2.plot(range(len(cumsum)), cumsum / (np.arange(len(cumsum)) + 1) * 100)
ax2.set_xlabel('Frame Number')
ax2.set_ylabel('Cumulative Sync Rate (%)')
ax2.set_title('Sync Rate Over Time')
ax2.axhline(y=95, color='g', linestyle='--', label='95% target')
ax2.legend()

plt.tight_layout()
plt.show()

## 4. Memory Usage

In [ ]:
process = psutil.Process()

print("Current Memory Usage:")
mem_info = process.memory_info()
print(f"  RSS: {mem_info.rss / 1e6:.1f} MB")
print(f"  VMS: {mem_info.vms / 1e6:.1f} MB")

# System memory
sys_mem = psutil.virtual_memory()
print(f"\nSystem Memory:")
print(f"  Total: {sys_mem.total / 1e9:.1f} GB")
print(f"  Available: {sys_mem.available / 1e9:.1f} GB")
print(f"  Used: {sys_mem.percent}%")

## 5. Performance Summary

In [ ]:
print("\n" + "="*50)
print("Performance Summary")
print("="*50)

targets = [
    ('Inference FPS (640x640)', results[-1]['fps'] if 'results' in dir() else 0, 30, 'FPS'),
    ('Sync Rate', stats['sync_rate'] * 100, 95, '%'),
    ('Memory Usage', mem_info.rss / 1e6, 2000, 'MB')
]

for name, value, target, unit in targets:
    status = '✓' if (value >= target if unit != 'MB' else value <= target) else '✗'
    print(f"{status} {name}: {value:.1f} {unit} (target: {target} {unit})")